# Fraud Detection ML Pipeline
**Course:** IS455 | **Framework:** CRISP-DM | **Database:** shop.db

---

## Phase 1: Business Understanding

### Project Charter

**Objective:** Build a binary classifier to predict `is_fraud` on the `orders` table, enabling real-time fraud detection for the e-commerce platform.

**Business Problem:**  
Fraudulent orders result in chargebacks, goods shipped without payment, and reputational damage. **False negatives (missed fraud) are more costly than false positives** (legitimate orders flagged for review), which drives our metric priorities.

**Success Metrics (ranked by priority):**
1. **Recall (fraud class)** — minimize missed fraud; target ≥ 0.75
2. **F1-score (fraud class)** — balance precision/recall; target ≥ 0.65
3. **ROC-AUC** — overall discriminative ability; target ≥ 0.85
4. Accuracy is *not* a primary metric — the 6.36% fraud rate makes it misleading

**Output:** `model.sav` — serialized sklearn Pipeline for real-time scoring via Vercel API connected to Supabase backend.

In [ ]:
# Cell 2 — Imports and Path Setup
import sys
import os
sys.path.insert(0, os.path.abspath('..'))
import pyLibrary as pl

import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, f1_score, recall_score,
    precision_score, roc_auc_score, accuracy_score,
    roc_curve, precision_recall_curve, auc
)
from imblearn.over_sampling import SMOTE

pd.set_option('display.max_columns', None)
print('Imports complete.')

---

## Phase 2: Data Understanding

**Goal:** Understand the raw data, assess data quality, identify signals for fraud detection, and document any issues before modeling.

In [ ]:
# Cell 4 — Database Connection and Raw Data Load
conn = sqlite3.connect('shop.db')

df_orders_raw    = pd.read_sql_query('SELECT * FROM orders', conn)
df_customers_raw = pd.read_sql_query('SELECT * FROM customers', conn)
df_shipments_raw = pd.read_sql_query('SELECT * FROM shipments', conn)
df_items_raw     = pd.read_sql_query('SELECT * FROM order_items', conn)

print(f'orders      : {df_orders_raw.shape}')
print(f'customers   : {df_customers_raw.shape}')
print(f'shipments   : {df_shipments_raw.shape}')
print(f'order_items : {df_items_raw.shape}')
print()
print('orders columns:', list(df_orders_raw.columns))

In [ ]:
# Cell 5 — Class Balance Check
fraud_counts = df_orders_raw['is_fraud'].value_counts().sort_index()
fraud_pct    = df_orders_raw['is_fraud'].value_counts(normalize=True).sort_index() * 100

print('Fraud class distribution:')
print(pd.DataFrame({'count': fraud_counts, 'percent (%)': fraud_pct.round(2)}))

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['Legitimate (0)', 'Fraud (1)'], fraud_counts.values,
               color=['steelblue', 'tomato'], edgecolor='white')
for bar, count, pct in zip(bars, fraud_counts.values, fraud_pct.values):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 30,
            f'{count}\n({pct:.1f}%)', ha='center', fontsize=11)
ax.set_title('Target Class Distribution: is_fraud')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 6 — Tabular Univariate Stats for Orders
stats_table = pl.unistats(df_orders_raw)
display(stats_table)

In [ ]:
# Cell 7 — Distribution Plots for Key Numeric Columns
key_cols = ['order_subtotal', 'order_total', 'risk_score', 'shipping_fee', 'tax_amount']
pl.univariate(df_orders_raw[key_cols])

In [ ]:
# Cell 8 — Bivariate Analysis vs is_fraud
orders_model_cols = df_orders_raw.drop(columns=['order_id', 'customer_id'])
corr_df = pl.bivariate(orders_model_cols, 'is_fraud')
print('\nNumeric correlations with is_fraud:')
display(corr_df)

# Correlation heatmap — numeric columns only
pl.correlation_heatmap(df_orders_raw.select_dtypes('number'))

In [ ]:
# Cell 9 — Customer and Shipment Table Profiling
print('=== CUSTOMERS ===')
display(pl.unistats(df_customers_raw))

print('\n=== SHIPMENTS ===')
display(pl.unistats(df_shipments_raw))

# Cross-table signal: fraud rate by late_delivery
df_ship_fraud = pd.read_sql_query("""
    SELECT s.late_delivery, o.is_fraud
    FROM orders o
    JOIN shipments s ON o.order_id = s.order_id
""", conn)
print('\nFraud rate by late_delivery:')
print(df_ship_fraud.groupby('late_delivery')['is_fraud'].agg(['mean', 'count']).round(4))

# Fraud rate by payment_method
print('\nFraud rate by payment_method:')
print(df_orders_raw.groupby('payment_method')['is_fraud'].agg(['mean', 'count']).sort_values('mean', ascending=False).round(4))

# Fraud rate by ip_country
print('\nFraud rate by ip_country:')
print(df_orders_raw.groupby('ip_country')['is_fraud'].agg(['mean', 'count']).sort_values('mean', ascending=False).round(4))

### Data Understanding Summary

**Key Findings:**
1. **Class imbalance:** ~6.36% fraud rate — accuracy is misleading; use Recall and F1
2. **`risk_score`:** highest single predictor (mean ~54.5 for fraud vs ~24.2 for legitimate)
3. **`late_delivery`:** ~10.6% fraud rate vs ~0.05% for on-time shipments
4. **`billing_zip != shipping_zip`:** ~11.7% fraud rate vs ~5.9% when matched
5. **High-value orders (>$500):** ~12.1% fraud vs ~3.4% for lower-value
6. **`crypto` payment method:** highest fraud rate (~10.3%)
7. **International IP (non-US):** slightly elevated fraud rate
8. **`promo_code`:** 74.78% null — use `promo_used` binary flag; drop `promo_code`

**Data Quality Issues:**
- `promo_code`: 74.78% null → use existing `promo_used` binary column instead
- `account_age_days` may be negative (order placed before customer `created_at`) — data anomaly, preserved as fraud signal
- Datetime columns stored as TEXT strings → parse with `pd.to_datetime()`

---

## Phase 3: Data Preparation

**Steps:**
1. SQL join: merge orders + customers + shipments + order_items aggregations
2. Feature engineering (10 domain-driven features)
3. Clean and wrangle
4. Handle class imbalance
5. Encode and split

In [ ]:
# Cell 12 — Master SQL Join
query = """
SELECT
    -- Order base features
    o.order_id,
    o.customer_id,
    o.order_datetime,
    o.billing_zip,
    o.shipping_zip,
    o.shipping_state,
    o.payment_method,
    o.device_type,
    o.ip_country,
    o.promo_used,
    o.order_subtotal,
    o.shipping_fee,
    o.tax_amount,
    o.order_total,
    o.risk_score,
    o.is_fraud,

    -- Customer features
    c.gender,
    c.birthdate,
    c.created_at          AS customer_created_at,
    c.customer_segment,
    c.loyalty_tier,
    c.is_active           AS customer_is_active,

    -- Shipment features
    s.carrier,
    s.shipping_method,
    s.distance_band,
    s.late_delivery,

    -- Per-order item aggregations
    oi_agg.item_count,
    oi_agg.total_items,
    oi_agg.avg_unit_price,
    oi_agg.unique_products

FROM orders o
LEFT JOIN customers c
    ON o.customer_id = c.customer_id
LEFT JOIN shipments s
    ON o.order_id = s.order_id
LEFT JOIN (
    SELECT
        order_id,
        COUNT(order_item_id)       AS item_count,
        SUM(quantity)              AS total_items,
        AVG(unit_price)            AS avg_unit_price,
        COUNT(DISTINCT product_id) AS unique_products
    FROM order_items
    GROUP BY order_id
) oi_agg
    ON o.order_id = oi_agg.order_id

ORDER BY o.order_id
"""

df = pd.read_sql_query(query, conn)
conn.close()

print(f'Joined dataset shape: {df.shape}')
print(f'\nNull counts (non-zero only):')
null_counts = df.isnull().sum()
print(null_counts[null_counts > 0])

In [ ]:
# Cell 13 — Feature Engineering

# --- Feature 1: ZIP code mismatch flag ---
df['zip_mismatch'] = (df['billing_zip'] != df['shipping_zip']).astype(int)

# --- Feature 2: International IP flag ---
df['ip_international'] = (df['ip_country'] != 'US').astype(int)

# --- Feature 3: High-risk country flag ---
df['ip_high_risk_country'] = df['ip_country'].isin(['NG', 'IN', 'BR']).astype(int)

# --- Feature 4: Parse datetimes ---
df['order_dt_parsed']           = pd.to_datetime(df['order_datetime'], errors='coerce')
df['customer_created_at_parsed'] = pd.to_datetime(df['customer_created_at'], errors='coerce')
df['birthdate_parsed']           = pd.to_datetime(df['birthdate'], errors='coerce')

# --- Feature 5: Order time-of-day features ---
df['order_hour']       = df['order_dt_parsed'].dt.hour
df['order_day_of_week'] = df['order_dt_parsed'].dt.dayofweek  # 0=Monday, 6=Sunday
df['order_month']      = df['order_dt_parsed'].dt.month
df['is_weekend']       = (df['order_day_of_week'] >= 5).astype(int)
df['is_night_order']   = ((df['order_hour'] >= 23) | (df['order_hour'] <= 4)).astype(int)

# --- Feature 6: Account age in days ---
df['account_age_days'] = (
    df['order_dt_parsed'] - df['customer_created_at_parsed']
).dt.days
# Negative means order placed before account created — anomaly and fraud signal
df['order_before_account']   = (df['account_age_days'] < 0).astype(int)
df['account_age_days_capped'] = df['account_age_days'].clip(lower=0)

# --- Feature 7: Customer age at time of order ---
df['customer_age_years'] = (
    df['order_dt_parsed'] - df['birthdate_parsed']
).dt.days // 365

# --- Feature 8: High-value order flag ---
df['is_high_value'] = (df['order_total'] > 500).astype(int)

# --- Feature 9: Promo used on high-value order (interaction) ---
df['promo_high_value'] = ((df['promo_used'] == 1) & (df['is_high_value'] == 1)).astype(int)

# --- Feature 10: Product diversity ratio ---
df['product_diversity_ratio'] = (
    df['unique_products'] / df['item_count'].replace(0, 1)
)

# Drop raw datetime strings and intermediate parse columns
drop_cols = [
    'order_datetime', 'customer_created_at', 'birthdate',
    'order_dt_parsed', 'customer_created_at_parsed', 'birthdate_parsed',
    'billing_zip', 'shipping_zip',   # replaced by zip_mismatch
    'account_age_days'               # replaced by _capped + order_before_account
]
df.drop(columns=drop_cols, inplace=True, errors='ignore')

print('Feature engineering complete.')
print(f'Dataset shape after engineering: {df.shape}')
new_features = [
    'zip_mismatch', 'ip_international', 'ip_high_risk_country',
    'order_hour', 'order_day_of_week', 'order_month', 'is_weekend', 'is_night_order',
    'order_before_account', 'account_age_days_capped', 'customer_age_years',
    'is_high_value', 'promo_high_value', 'product_diversity_ratio'
]
print('New features:', new_features)

In [ ]:
# Cell 14 — Basic Wrangling: Drop ID columns and near-constant/quasi-unique columns
# Drop order_id and customer_id manually first (identifiers, not features)
df_model = df.drop(columns=['order_id', 'customer_id'], errors='ignore')

# basic_wrangling drops columns with >= 95% missing or >= 95% unique values
df_model = pl.basic_wrangling(df_model)

print(f'\nShape after basic_wrangling: {df_model.shape}')
print('Remaining columns:', list(df_model.columns))

In [ ]:
# Cell 15 — Bin Rare Categories
# ip_country and shipping_state have high cardinality with many rare values
df_model = pl.bin_rare_categories(
    df_model,
    cols=['ip_country', 'shipping_state'],
    min_prop=0.05
)

# Drop originals, keep binned versions
df_model.drop(columns=['ip_country', 'shipping_state'], inplace=True, errors='ignore')
df_model.rename(columns={
    'ip_country_binned'    : 'ip_country',
    'shipping_state_binned': 'shipping_state'
}, inplace=True)

print('ip_country value counts after binning:')
print(df_model['ip_country'].value_counts())
print('\nshipping_state value counts (top 10):')
print(df_model['shipping_state'].value_counts().head(10))

In [ ]:
# Cell 16 — Missing Value Handling
print('Missing values before imputation:')
null_before = df_model.isnull().sum()
print(null_before[null_before > 0] if null_before.any() else 'None')

# missing_fill handles numeric columns via IterativeImputer (dataset < 200k rows)
# Drops rows where target is null (none expected)
df_model = pl.missing_fill(df_model, target='is_fraud', mar='drop')

print(f'\nShape after missing_fill: {df_model.shape}')
print(f'Remaining missing values: {df_model.isnull().sum().sum()}')

In [ ]:
# Cell 17 — Outlier Capping with IQR Winsorization
# Cap monetary, count, and age columns — do NOT cap risk_score (calibrated 0-100 scale)
numeric_to_cap = [
    'order_subtotal', 'shipping_fee', 'tax_amount', 'order_total',
    'avg_unit_price', 'account_age_days_capped', 'customer_age_years',
    'product_diversity_ratio', 'item_count', 'total_items', 'unique_products'
]
# Filter to only columns that actually exist
numeric_to_cap = [c for c in numeric_to_cap if c in df_model.columns]

df_model = pl.cap_outliers_iqr(df_model, cols=numeric_to_cap)
print('Outlier capping complete.')
print(f'Dataset shape: {df_model.shape}')

In [ ]:
# Cell 18 — Stratified Train/Test Split
# stratify=True preserves the 6.36% fraud rate in both train and test sets
X_train, X_test, y_train, y_test = pl.split_data(
    df_model,
    target='is_fraud',
    test_size=0.2,
    random_state=42,
    stratify=True
)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Train fraud rate: {y_train.mean():.4f} ({y_train.sum()} fraud cases)')
print(f'Test  fraud rate: {y_test.mean():.4f}  ({y_test.sum()} fraud cases)')

In [ ]:
# Cell 19 — Encode Features and Apply SMOTE (for non-pipeline models)
# Pipeline models (LR, DT, RF) receive raw X_train — encoding handled internally
# Gradient Boosting needs encoded + SMOTE-resampled data (no class_weight support)

# Identify categorical columns
cat_cols_to_encode = X_train.select_dtypes(include='object').columns.tolist()
print('Categorical columns for encoding:', cat_cols_to_encode)

# One-hot encode (drop_first=True avoids dummy variable trap)
X_train_encoded = pl.encode_features(X_train, cat_cols=cat_cols_to_encode,
                                      strategy='onehot', drop_first=True)
X_test_encoded  = pl.encode_features(X_test, cat_cols=cat_cols_to_encode,
                                      strategy='onehot', drop_first=True)

# Align columns — test set may be missing dummy columns if rare categories absent
X_train_encoded, X_test_encoded = X_train_encoded.align(
    X_test_encoded, join='left', axis=1, fill_value=0
)

# Convert any bool columns to int (pd.get_dummies creates bool in newer pandas)
bool_cols = X_train_encoded.select_dtypes(include='bool').columns
X_train_encoded[bool_cols] = X_train_encoded[bool_cols].astype(int)
X_test_encoded[bool_cols]  = X_test_encoded[bool_cols].astype(int)

# Apply SMOTE only to training data (never test data — prevents leakage)
# sampling_strategy=0.25 raises fraud proportion from ~6.4% to ~20%
smote = SMOTE(sampling_strategy=0.25, random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(
    X_train_encoded.astype(float), y_train
)

print(f'\nBefore SMOTE — fraud: {y_train.sum()}, legit: {(y_train == 0).sum()}')
print(f'After SMOTE  — fraud: {y_train_smote.sum()}, legit: {(y_train_smote == 0).sum()}')
print(f'Post-SMOTE fraud rate: {y_train_smote.mean():.4f}')

---

## Phase 4: Modeling

Four classifiers are trained across two imbalance-handling strategies:

| Model | Imbalance Strategy | Input |
|---|---|---|
| Logistic Regression | `class_weight='balanced'` | Raw `X_train` (pipeline encodes) |
| Decision Tree | `class_weight='balanced'` | Raw `X_train` (pipeline encodes) |
| Random Forest | `class_weight='balanced'` | Raw `X_train` (pipeline encodes) |
| Gradient Boosting | SMOTE oversampling | `X_train_smote` (pre-encoded) |

All pipeline models use `pl.make_pipeline_for_model()` which wraps a `ColumnTransformer` (median-impute + scale numeric; mode-impute + one-hot categorical) to prevent data leakage.

In [ ]:
# Cell 21 — Confirm Pipeline Step Names (important for tune_random/tune_grid)
# make_pipeline_for_model creates steps: ("preprocess", ColumnTransformer), ("model", estimator)
# Hyperparameter grid keys for pipeline models must use "model__" prefix
_demo_pipe = pl.make_pipeline_for_model(X_train, LogisticRegression())
print('Pipeline steps:', [name for name, _ in _demo_pipe.steps])
print('Param prefix for tuning: model__<param_name>')
del _demo_pipe

In [ ]:
# Cell 22 — Model 1: Logistic Regression (class_weight='balanced')
lr_model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)
pipeline_lr = pl.make_pipeline_for_model(X_train, lr_model)

lr_results = pl.eval_classification(
    name='Logistic Regression (balanced)',
    model=pipeline_lr,
    X_train=X_train, y_train=y_train,
    X_test=X_test, y_test=y_test,
    fit=True
)

In [ ]:
# Cell 23 — Model 2: Decision Tree (interpretability baseline)
dt_model = DecisionTreeClassifier(
    class_weight='balanced',
    max_depth=6,
    min_samples_leaf=10,
    random_state=42
)
pipeline_dt = pl.make_pipeline_for_model(X_train, dt_model)

dt_results = pl.eval_classification(
    name='Decision Tree (balanced, max_depth=6)',
    model=pipeline_dt,
    X_train=X_train, y_train=y_train,
    X_test=X_test, y_test=y_test,
    fit=True
)

In [ ]:
# Cell 24 — Model 3: Random Forest (class_weight='balanced')
rf_model = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    max_features='sqrt',
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)
pipeline_rf = pl.make_pipeline_for_model(X_train, rf_model)

rf_results = pl.eval_classification(
    name='Random Forest (balanced, 200 trees)',
    model=pipeline_rf,
    X_train=X_train, y_train=y_train,
    X_test=X_test, y_test=y_test,
    fit=True
)

In [ ]:
# Cell 25 — Model 4: Gradient Boosting (SMOTE — no class_weight support)
gb_model = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    random_state=42
)
# Fit directly on SMOTE-resampled encoded data (pre-encoded, not a pipeline)
gb_model.fit(X_train_smote, y_train_smote)

gb_results = pl.eval_classification(
    name='Gradient Boosting (SMOTE, 200 trees)',
    model=gb_model,
    X_train=X_train_smote, y_train=y_train_smote,
    X_test=X_test_encoded.astype(float), y_test=y_test,
    fit=False  # Already fitted above
)

In [ ]:
# Cell 26 — Cross-Validation (5-fold Stratified, scoring=roc_auc)
print('=== Cross-Validation (5-fold Stratified, ROC-AUC) ===\n')

cv_scores = {}

for name, pipeline in [
    ('Logistic Regression', pipeline_lr),
    ('Decision Tree',       pipeline_dt),
    ('Random Forest',       pipeline_rf)
]:
    print(f'--- {name} ---')
    scores = pl.cross_validate_model(
        model=pipeline, X=X_train, y=y_train,
        cv=5, scoring='roc_auc', stratified=True
    )
    cv_scores[name] = {'mean_auc': round(scores.mean(), 4), 'std_auc': round(scores.std(), 4)}

print('--- Gradient Boosting (SMOTE) ---')
scores_gb = pl.cross_validate_model(
    model=gb_model, X=X_train_encoded.astype(float), y=y_train,
    cv=5, scoring='roc_auc', stratified=True
)
cv_scores['Gradient Boosting'] = {'mean_auc': round(scores_gb.mean(), 4), 'std_auc': round(scores_gb.std(), 4)}

print('\nCV Summary:')
display(pd.DataFrame(cv_scores).T.sort_values('mean_auc', ascending=False))

In [ ]:
# Cell 27 — Hyperparameter Tuning: Random Forest (RandomizedSearchCV)
# NOTE: param keys use 'model__' prefix because make_pipeline_for_model
#       names the classifier step 'model'
from scipy.stats import randint

rf_param_dist = {
    'model__n_estimators'   : randint(100, 400),
    'model__max_depth'      : [None, 4, 6, 8, 10],
    'model__min_samples_leaf': randint(2, 15),
    'model__max_features'   : ['sqrt', 'log2'],
    'model__class_weight'   : ['balanced', 'balanced_subsample']
}

print('Tuning Random Forest (n_iter=30, 5-fold CV)...')
best_rf, rf_search = pl.tune_random(
    model=pipeline_rf,
    param_dist=rf_param_dist,
    X_train=X_train, y_train=y_train,
    n_iter=30, cv=5, scoring='roc_auc'
)

In [ ]:
# Cell 28 — Hyperparameter Tuning: Gradient Boosting (GridSearchCV)
gb_param_grid = {
    'n_estimators' : [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth'    : [3, 4, 5],
    'subsample'    : [0.7, 0.8, 1.0]
}

print('Tuning Gradient Boosting (GridSearchCV, 5-fold CV)...')
best_gb, gb_search = pl.tune_grid(
    model=GradientBoostingClassifier(random_state=42),
    param_grid=gb_param_grid,
    X_train=X_train_smote, y_train=y_train_smote,
    cv=5, scoring='roc_auc'
)

# Re-evaluate tuned models
print('\n--- Tuned RF Evaluation ---')
tuned_rf_results = pl.eval_classification(
    'Random Forest (tuned)', best_rf,
    X_train, y_train, X_test, y_test, fit=False
)

print('\n--- Tuned GB Evaluation ---')
tuned_gb_results = pl.eval_classification(
    'Gradient Boosting (tuned)', best_gb,
    X_train_smote, y_train_smote,
    X_test_encoded.astype(float), y_test, fit=False
)

In [ ]:
# Cell 29 — Feature Importance Plot (Tuned Random Forest)
rf_classifier  = best_rf.named_steps['model']
preprocessor   = best_rf.named_steps['preprocess']

try:
    feature_names = preprocessor.get_feature_names_out()
    # Clean up prefixes added by ColumnTransformer
    feature_names = [n.replace('num__', '').replace('cat__', '') for n in feature_names]
except Exception:
    feature_names = [f'feature_{i}' for i in range(len(rf_classifier.feature_importances_))]

importances = rf_classifier.feature_importances_
top_n = min(20, len(importances))
indices = np.argsort(importances)[::-1][:top_n]

plt.figure(figsize=(10, 7))
plt.barh(range(top_n), importances[indices][::-1],
         color='steelblue', edgecolor='white', align='center')
plt.yticks(range(top_n), [feature_names[i] for i in indices][::-1])
plt.xlabel('Feature Importance')
plt.title('Random Forest (Tuned): Top Feature Importances')
plt.tight_layout()
plt.show()

---

## Phase 5: Evaluation

**Primary metrics:** Recall (fraud class) and F1-score (fraud class)  
**Secondary:** ROC-AUC, Precision-Recall AUC  
All evaluation is performed on the held-out test set (20%, stratified).

> **Note:** `pl.eval_classification` reports weighted-average F1. The table below uses `f1_score(pos_label=1, average='binary')` — the fraud-class F1 that matters for the business.

In [ ]:
# Cell 31 — Fraud-Class Metric Comparison Table
def fraud_metrics(name, model, X_eval, y_eval):
    """Compute fraud-class specific metrics (pos_label=1)."""
    y_pred = model.predict(X_eval)
    y_prob = model.predict_proba(X_eval)[:, 1] if hasattr(model, 'predict_proba') else None
    return {
        'Model'             : name,
        'Recall (fraud)'    : round(recall_score(y_eval, y_pred, pos_label=1, zero_division=0), 4),
        'Precision (fraud)' : round(precision_score(y_eval, y_pred, pos_label=1, zero_division=0), 4),
        'F1 (fraud)'        : round(f1_score(y_eval, y_pred, pos_label=1, zero_division=0), 4),
        'ROC-AUC'           : round(roc_auc_score(y_eval, y_prob), 4) if y_prob is not None else 'N/A',
        'Accuracy'          : round(accuracy_score(y_eval, y_pred), 4)
    }

X_test_enc = X_test_encoded.astype(float)

results_table = pd.DataFrame([
    fraud_metrics('Logistic Regression (balanced)', pipeline_lr, X_test, y_test),
    fraud_metrics('Decision Tree (balanced)',        pipeline_dt, X_test, y_test),
    fraud_metrics('Random Forest (baseline)',        pipeline_rf, X_test, y_test),
    fraud_metrics('Gradient Boosting (baseline)',    gb_model,    X_test_enc, y_test),
    fraud_metrics('Random Forest (tuned)',           best_rf,     X_test, y_test),
    fraud_metrics('Gradient Boosting (tuned)',       best_gb,     X_test_enc, y_test),
])

results_table = results_table.sort_values('Recall (fraud)', ascending=False).reset_index(drop=True)
display(results_table)

In [ ]:
# Cell 32 — ROC Curves and Precision-Recall Curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

models_to_plot = {
    'Logistic Regression'       : (pipeline_lr, X_test),
    'Decision Tree'             : (pipeline_dt, X_test),
    'Random Forest (tuned)'     : (best_rf,     X_test),
    'Gradient Boosting (tuned)' : (best_gb,     X_test_enc),
}

for name, (model, X_eval) in models_to_plot.items():
    y_prob = model.predict_proba(X_eval)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    pr_vals, rc_vals, _ = precision_recall_curve(y_test, y_prob)
    roc_auc_val = auc(fpr, tpr)
    pr_auc_val  = auc(rc_vals, pr_vals)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={roc_auc_val:.3f})')
    axes[1].plot(rc_vals, pr_vals, label=f'{name} (AUC={pr_auc_val:.3f})')

axes[0].plot([0, 1], [0, 1], 'k--', linewidth=0.8)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves')
axes[0].legend(fontsize=8)

axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curves')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 33 — Confusion Matrices for Top 2 Models
pl.plot_confusion_matrix(
    best_rf, X_test, y_test,
    labels=[0, 1],
    title='Random Forest (Tuned) — Confusion Matrix'
)

pl.plot_confusion_matrix(
    best_gb, X_test_enc, y_test,
    labels=[0, 1],
    title='Gradient Boosting (Tuned) — Confusion Matrix'
)

In [ ]:
# Cell 34 — Threshold Optimization
# Lower threshold below 0.5 to maximize Recall while keeping F1 above floor
# Evaluate both finalists; pick the one with higher fraud-class F1 at Recall >= 0.75

def optimize_threshold(name, model, X_eval, y_eval, recall_floor=0.75):
    """Find threshold maximizing F1 subject to Recall >= recall_floor."""
    y_prob = model.predict_proba(X_eval)[:, 1]
    precisions, recalls, thresholds = precision_recall_curve(y_eval, y_prob)
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)

    valid_mask = recalls[:-1] >= recall_floor
    if valid_mask.any():
        best_idx      = f1_scores[:-1][valid_mask].argmax()
        best_threshold = thresholds[valid_mask][best_idx]
        best_recall    = recalls[:-1][valid_mask][best_idx]
        best_precision = precisions[:-1][valid_mask][best_idx]
        best_f1        = f1_scores[:-1][valid_mask][best_idx]
    else:
        best_threshold = 0.5
        best_recall = best_precision = best_f1 = float('nan')
        print(f'WARNING: {name} could not achieve Recall >= {recall_floor}')

    print(f'\n{name}')
    print(f'  Optimal threshold: {best_threshold:.3f}')
    print(f'  Recall={best_recall:.4f}  Precision={best_precision:.4f}  F1={best_f1:.4f}')

    # Plot threshold curve
    plt.figure(figsize=(9, 4))
    plt.plot(thresholds, precisions[:-1], label='Precision')
    plt.plot(thresholds, recalls[:-1],    label='Recall')
    plt.plot(thresholds, f1_scores[:-1],  label='F1')
    plt.axvline(x=best_threshold, color='red', linestyle='--',
                label=f'Optimal: {best_threshold:.2f}')
    plt.axhline(y=recall_floor, color='gray', linestyle=':', alpha=0.5,
                label=f'Recall floor: {recall_floor}')
    plt.xlabel('Classification Threshold')
    plt.ylabel('Score')
    plt.title(f'Threshold Curve — {name}')
    plt.legend(fontsize=9)
    plt.tight_layout()
    plt.show()

    return best_threshold

thresh_rf = optimize_threshold('Random Forest (tuned)',        best_rf, X_test,    y_test)
thresh_gb = optimize_threshold('Gradient Boosting (tuned)',    best_gb, X_test_enc, y_test)

In [ ]:
# Cell 35 — Final Model Selection
# Selection criteria (in order):
#   1. Recall (fraud class) >= 0.75 at optimal threshold
#   2. F1 (fraud class) — highest among qualifying models
#   3. ROC-AUC as tiebreaker

def apply_threshold(model, X_eval, threshold):
    """Apply a custom probability threshold for classification."""
    y_prob = model.predict_proba(X_eval)[:, 1]
    return (y_prob >= threshold).astype(int)

# Evaluate both finalists at their optimal thresholds
final_models = {
    'Random Forest (tuned)': {
        'model': best_rf, 'X_eval': X_test,
        'threshold': thresh_rf, 'X_train': X_train, 'y_train': y_train
    },
    'Gradient Boosting (tuned)': {
        'model': best_gb, 'X_eval': X_test_enc,
        'threshold': thresh_gb, 'X_train': X_train_smote, 'y_train': y_train_smote
    }
}

threshold_results = []
for name, info in final_models.items():
    y_pred_t = apply_threshold(info['model'], info['X_eval'], info['threshold'])
    y_prob   = info['model'].predict_proba(info['X_eval'])[:, 1]
    threshold_results.append({
        'Model'             : name,
        'Threshold'         : round(info['threshold'], 3),
        'Recall (fraud)'    : round(recall_score(y_test, y_pred_t, pos_label=1, zero_division=0), 4),
        'Precision (fraud)' : round(precision_score(y_test, y_pred_t, pos_label=1, zero_division=0), 4),
        'F1 (fraud)'        : round(f1_score(y_test, y_pred_t, pos_label=1, zero_division=0), 4),
        'ROC-AUC'           : round(roc_auc_score(y_test, y_prob), 4)
    })

threshold_df = pd.DataFrame(threshold_results).sort_values('F1 (fraud)', ascending=False)
display(threshold_df)

# Assign final model based on results (highest F1 with Recall >= 0.75)
best_row = threshold_df.iloc[0]
final_name = best_row['Model']
final_threshold = best_row['Threshold']

if final_name == 'Random Forest (tuned)':
    final_model = best_rf
    final_X_test = X_test
else:
    final_model = best_gb
    final_X_test = X_test_enc

print(f'\nFinal Model Selected: {final_name}')
print(f'Production Threshold: {final_threshold}')

In [ ]:
# Cell 36 — Learning Curve: Bias/Variance Diagnosis
pl.plot_learning_curve(
    model=best_rf,
    X=X_train,
    y=y_train,
    cv=5,
    scoring='roc_auc'
)

---

## Phase 6: Deployment

The final model is serialized to `model.sav` using pickle for integration with the Supabase + Vercel production stack.

In [ ]:
# Cell 38 — Serialize Final Model to model.sav
model_path = 'model.sav'

pickle.dump(final_model, open(model_path, 'wb'))

# Verify: reload and run 5 sample predictions
loaded_model = pickle.load(open(model_path, 'rb'))
sample_preds = loaded_model.predict(final_X_test[:5])
sample_probs = loaded_model.predict_proba(final_X_test[:5])[:, 1]

print(f'Model serialized to: {model_path}')
print(f'Model type: {type(loaded_model).__name__}')
print(f'Sample predictions : {sample_preds}')
print(f'Sample probabilities: {sample_probs.round(4)}')
print(f'Production threshold: {final_threshold}')
print(f'Usage: flag as fraud if predict_proba(X)[:, 1] > {final_threshold}')

### Deployment Integration Notes

#### Feature Engineering Requirements for Production API

When scoring new orders in real-time, the Vercel serverless endpoint must:

1. **Receive:** raw order record + `customer_id`
2. **Query Supabase for:** customer record (`created_at`, `birthdate`, `loyalty_tier`, etc.) and order_items (`item_count`, `unique_products`, etc.)
3. **Engineer the same features as Phase 3:**
   - `zip_mismatch`, `ip_international`, `ip_high_risk_country`
   - `order_hour`, `order_day_of_week`, `order_month`, `is_weekend`, `is_night_order`
   - `account_age_days_capped`, `order_before_account`, `customer_age_years`
   - `is_high_value`, `promo_high_value`, `product_diversity_ratio`
4. **Score:** `probability = model.predict_proba(X_new)[0][1]`
5. **Flag:** `is_fraud_predicted = int(probability > THRESHOLD)`

#### Production Limitations

> **`late_delivery`** is used at training time (historical data) but is **not available at order-creation time**. In production, either (a) exclude it from the feature set and retrain, or (b) score orders twice — once at creation and once after shipment resolves.

#### Retraining Triggers
- Retrain monthly or when fraud rate shifts by more than 2 percentage points
- If production Recall drops below 0.65, retrain immediately
- Use stratified sampling to maintain class distribution in retraining data